# AMPS I · Week 14 — HW13: reproduce the DFG 2D-3 benchmark
**Due:** before the Week 15 lecture · **Points:** 100 · **Submit:** this notebook (`.ipynb`) with all outputs

**Rules**
- Before submitting: *Runtime → Restart session and run all*. Notebooks that do not run end to end receive 0.
- An error inside a `%%fenicsx` cell does **not** stop *Run all*. Read every output before you submit.
- LLM assistants are encouraged. Fill in the **AI-usage log** at the end.
- Every number you report must come from a cell you ran.
- **Run time.** Q2–Q4 together take about 20 minutes on Colab (measured: Q2 ≈ 8.5, Q3 ≈ 6; Q4 roughly 5). Start early; save results to `/tmp/fnx/` so that
  plots do not re-run simulations; save the notebook only after the last cell has finished.

**Benchmark** (Schäfer & Turek 1996; reference values John 2004): channel $2.2\times0.41$, cylinder $r=0.05$ at $(0.2,0.2)$, $\nu=10^{-3}$, $\rho=1$,
$U(t)=1.5\sin(\pi t/8)$ at the inlet centre (parabolic profile), $T=8$. $c_{D,\max}=2.950921575$ at $t=3.93625$, $c_{L,\max}=0.47795$ at $t=5.693125$, $\Delta p(8)=-0.1116$.

| Question | Topic | Points |
|---|---|:---:|
| Q1 | Projection methods, IPCS, CFL and the cell Reynolds number | 20 |
| Q2 | DFG 2D-3 on two meshes | 40 |
| Q3 | Time step: stability vs accuracy | 20 |
| Q4 | LLM debugging: a DFG script with four bugs | 20 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Q1 — Theory (20 pts)

**(a) (8) Chorin.** Given $\mathbf u^*$ (no pressure in the momentum step), the projection step solves $\mathbf u^{n+1}-\mathbf u^*=-\Delta t\nabla p^{n+1}$, $\nabla\cdot\mathbf u^{n+1}=0$,
$\mathbf u^{n+1}\cdot\mathbf n=\mathbf g\cdot\mathbf n$ on $\partial\Omega$. Derive the pressure equation and its boundary condition. Show that $\mathbf u^{n+1}$ is the $L^2$-orthogonal projection of $\mathbf u^*$
onto divergence-free fields (Helmholtz decomposition). Why is the boundary condition for $p$ unphysical, and what does that cost in accuracy?

**(b) (6) IPCS.** Write the three sub-steps of the incremental scheme with $\phi=p^{n+1}-p^n$. Show that for a **steady** solution of the discrete scheme
($\mathbf u^{n+1}=\mathbf u^n$, $p^{n+1}=p^n$) the velocity and pressure satisfy the discrete steady Navier–Stokes equations exactly, and that Chorin's steady state does not.
Connect with the channel results of the lab.

**(c) (6) Time step.** Define the CFL number and the cell Reynolds number $\mathrm{Re}_h$. For the DFG coarse and medium meshes ($h_{\min}=r/3$, $r/6$) and $|\mathbf u|\le2$,
compute $\Delta t$ for CFL = 1 and $\mathrm{Re}_h$ at the cylinder. Why can the semi-implicit scheme (convected velocity implicit) run at CFL > 1, while a fully explicit
treatment cannot — and why is CFL ≲ 1 still a good rule for accuracy?

> *your answers to (a)–(c)*

## Q2 — DFG 2D-3 on two meshes (40 pts)

Use the lab's `dfg_mesh`/`dfg_run` (or your own IPCS — then verify it reproduces the lab's coarse numbers).

**(a) (15)** Run the **coarse** and the **medium** mesh with $\Delta t=1/400$. Table: cells, DOFs, wall time, max CFL, Krylov iterations per sub-step, $c_{D,\max}$ and its time,
$c_{L,\max}$ and its time, $\Delta p(8)$, and the relative errors with respect to the reference.

**(b) (10)** Plot $c_D(t)$ and $c_L(t)$ of both runs with the reference maxima.

**(c) (15)** Which quantity is most sensitive to the mesh, and why (boundary layer on the cylinder, onset of vortex shedding)? Which error remains on the medium mesh, is it
spatial or temporal (use Q3)? Estimate the cost of halving $h$ again (DOFs, steps, time).

In [ ]:
%%fenicsx -np 2
# TODO: Q2(a) coarse and medium, dt = 1/400; save the time series to /tmp/fnx/hw13_q2.npz

from mpi4py import MPI
from petsc4py import PETSc
import time
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2).tolist()

def lu(A):
    s = PETSc.KSP().create(comm); s.setOperators(A); s.setType("preonly")
    s.getPC().setType("lu"); s.getPC().setFactorSolverType("mumps")
    return s

def rhs(b, L, a=None, bcs=None):
    """Assemble L into b (zeroed first), lift and set the BCs of a, sum the ghost contributions."""
    with b.localForm() as loc:
        loc.set(0)
    assemble_vector(b, L)
    if a is not None:
        apply_lifting(b, [a], [bcs])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    if bcs is not None:
        set_bc(b, bcs)

In [ ]:
# TODO: Q2(b) plots

> *your table and answers to (c)*

## Q3 — Time step: stability vs accuracy (20 pts)

**(a) (10)** Medium mesh, $\Delta t=1/100$ and $1/200$ (with Q2's $1/400$): max CFL, $c_{D,\max}$, $c_{L,\max}$ with errors. Is the scheme stable at CFL > 1?

**(b) (10)** Which quantity converges with $\Delta t$ and which does not change — what does that say about the split between temporal and spatial error on this mesh?
Which $\Delta t$ would you choose, and why?

In [ ]:
%%fenicsx -np 2
# TODO: Q3 dt = 1/100 and 1/200 on the medium mesh

from mpi4py import MPI
from petsc4py import PETSc
import time
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2).tolist()

def lu(A):
    s = PETSc.KSP().create(comm); s.setOperators(A); s.setType("preonly")
    s.getPC().setType("lu"); s.getPC().setFactorSolverType("mumps")
    return s

def rhs(b, L, a=None, bcs=None):
    """Assemble L into b (zeroed first), lift and set the BCs of a, sum the ghost contributions."""
    with b.localForm() as loc:
        loc.set(0)
    assemble_vector(b, L)
    if a is not None:
        apply_lifting(b, [a], [bcs])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    if bcs is not None:
        set_bc(b, bcs)

> *your table and answers to (a), (b)*

## Q4 — LLM debugging: a DFG script with four bugs (20 pts)

An LLM was asked: *"DOLFINx 0.11, -np 2: DFG 2D-3 with IPCS on the coarse mesh (`dfg_mesh` given), dt = 1/400, T = 6; print the maximal drag and lift coefficients."*
It produced the cell below (run it as given; it uses `dfg_mesh` from the lab, included in the cell).

**(a) (12)** Find the four bugs: line, what is wrong, how it shows in the output, how you found it. Note that the bugs hide each other — fix them one at a time.

**(b) (5)** Corrected cell: $c_{D,\max}$, $c_{L,\max}$ and their times (compare with Q2's coarse run up to $t=6$).

**(c) (3)** Which bug produces a plausible-looking but wrong number, and which check exposes it?

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
from petsc4py import PETSc
import time
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2).tolist()

def lu(A):
    s = PETSc.KSP().create(comm); s.setOperators(A); s.setType("preonly")
    s.getPC().setType("lu"); s.getPC().setFactorSolverType("mumps")
    return s

def rhs(b, L, a=None, bcs=None):
    """Assemble L into b (zeroed first), lift and set the BCs of a, sum the ghost contributions."""
    with b.localForm() as loc:
        loc.set(0)
    assemble_vector(b, L)
    if a is not None:
        apply_lifting(b, [a], [bcs])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    if bcs is not None:
        set_bc(b, bcs)
import gmsh
from dolfinx import geometry
from dolfinx.io.gmsh import model_to_mesh

L_, H, C, R = 2.2, 0.41, (0.2, 0.2), 0.05
REF = dict(cd=2.950921575, t_cd=3.93625, cl=0.47795, t_cl=5.693125, dp=-0.1116)


def dfg_mesh(res_min_factor, lc_max):
    """Gmsh mesh of the DFG channel: cells of size r/res_min_factor at the cylinder, lc_max*H far away. Facet tags 2 inlet, 3 outlet, 4 walls, 5 cylinder."""
    gmsh.initialize(); gmsh.option.setNumber("General.Terminal", 0)
    if comm.rank == 0:
        rect = gmsh.model.occ.addRectangle(0, 0, 0, L_, H); obst = gmsh.model.occ.addDisk(C[0], C[1], 0, R, R)
        gmsh.model.occ.cut([(2, rect)], [(2, obst)]); gmsh.model.occ.synchronize()
        vol = gmsh.model.getEntities(dim=2); gmsh.model.addPhysicalGroup(2, [vol[0][1]], 1)
        tags = {2: [], 3: [], 4: [], 5: []}
        for bnd in gmsh.model.getBoundary(vol, oriented=False):
            com = gmsh.model.occ.getCenterOfMass(bnd[0], bnd[1])
            if np.allclose(com, [0, H / 2, 0]): tags[2].append(bnd[1])
            elif np.allclose(com, [L_, H / 2, 0]): tags[3].append(bnd[1])
            elif np.isclose(com[1], H) or np.isclose(com[1], 0): tags[4].append(bnd[1])
            else: tags[5].append(bnd[1])
        for tg, ent in tags.items():
            gmsh.model.addPhysicalGroup(1, ent, tg)
        d = gmsh.model.mesh.field.add("Distance"); gmsh.model.mesh.field.setNumbers(d, "EdgesList", tags[5])
        th = gmsh.model.mesh.field.add("Threshold"); gmsh.model.mesh.field.setNumber(th, "IField", d)
        gmsh.model.mesh.field.setNumber(th, "LcMin", R / res_min_factor); gmsh.model.mesh.field.setNumber(th, "LcMax", lc_max * H)
        gmsh.model.mesh.field.setNumber(th, "DistMin", R); gmsh.model.mesh.field.setNumber(th, "DistMax", 2 * H)
        gmsh.model.mesh.field.setAsBackgroundMesh(th)
        gmsh.option.setNumber("Mesh.Algorithm", 8)
        gmsh.model.mesh.generate(2); gmsh.model.mesh.setOrder(2); gmsh.model.mesh.optimize("Netgen")
    md = model_to_mesh(gmsh.model, comm, 0, gdim=2); gmsh.finalize()
    return md.mesh, md.facet_tags
MESHES = {"coarse": (3.0, 0.25), "medium": (6.0, 0.1)}

msh, ft = dfg_mesh(*MESHES["coarse"])
V = fem.functionspace(msh, ("Lagrange", 2, (2,))); Q = fem.functionspace(msh, ("Lagrange", 1))
dt, T, mu, rho = 1 / 400, 6.0, 0.001, 1.0
t = 0.0

def inflow(x):
    val = np.zeros((2, x.shape[1])); val[0] = 4 * 1.5 * np.sin(t * np.pi / 8) * x[1] * (H - x[1]) / H**2
    return val
u_in = fem.Function(V); u_in.interpolate(inflow)
bcu = [fem.dirichletbc(u_in, fem.locate_dofs_topological(V, 1, ft.find(2))),
       fem.dirichletbc(np.zeros(2), fem.locate_dofs_topological(V, 1, ft.find(4)), V),
       fem.dirichletbc(np.zeros(2), fem.locate_dofs_topological(V, 1, ft.find(5)), V)]
bcp = [fem.dirichletbc(0.0, fem.locate_dofs_topological(Q, 1, ft.find(3)), Q)]

u, v = ufl.TrialFunction(V), ufl.TestFunction(V); p, q = ufl.TrialFunction(Q), ufl.TestFunction(Q)
u_, u_s, u_n, u_n1 = (fem.Function(V) for _ in range(4)); p_, phi = fem.Function(Q), fem.Function(Q)
k = fem.Constant(msh, dt); muc = fem.Constant(msh, mu); rhoc = fem.Constant(msh, rho)
U = 0.5 * (u + u_n)
F1 = (rhoc / k * ufl.dot(u - u_n, v) + rhoc * ufl.inner(ufl.dot(1.5 * u_n - 0.5 * u_n1, ufl.nabla_grad(U)), v)
      + muc * ufl.inner(ufl.grad(U), ufl.grad(v)) - p_ * ufl.div(v)) * ufl.dx
a1, L1 = fem.form(ufl.lhs(F1)), fem.form(ufl.rhs(F1))
a2 = fem.form(ufl.dot(ufl.grad(p), ufl.grad(q)) * ufl.dx); L2 = fem.form(-rhoc / k * ufl.div(u_s) * q * ufl.dx)
a3 = fem.form(rhoc * ufl.dot(u, v) * ufl.dx); L3 = fem.form((rhoc * ufl.dot(u_s, v) + k * ufl.dot(ufl.grad(phi), v)) * ufl.dx)

A1 = assemble_matrix(a1, bcs=bcu); A1.assemble()        # assemble once, outside the time loop (faster)
A2 = assemble_matrix(a2, bcs=bcp); A2.assemble(); A3 = assemble_matrix(a3); A3.assemble()
b1, b2, b3 = create_vector(V), create_vector(Q), create_vector(V)
s1 = PETSc.KSP().create(comm); s1.setOperators(A1); s1.setType("bcgs"); s1.getPC().setType("jacobi")
s2 = PETSc.KSP().create(comm); s2.setOperators(A2); s2.setType("minres"); s2.getPC().setType("hypre"); s2.getPC().setHYPREType("boomeramg")
s3 = PETSc.KSP().create(comm); s3.setOperators(A3); s3.setType("cg"); s3.getPC().setType("sor")
for s in (s1, s2, s3):
    s.setTolerances(rtol=1e-8)

n = ufl.FacetNormal(msh)
dS = ufl.Measure("ds", domain=msh, subdomain_data=ft, subdomain_id=5)
u_t = ufl.inner(ufl.as_vector((n[1], -n[0])), u_)
drag = fem.form(2 / 0.1 * (muc / rhoc * ufl.inner(ufl.grad(u_t), n) * n[1] - p_ * n[0]) * dS)
lift = fem.form(-2 / 0.1 * (muc / rhoc * ufl.inner(ufl.grad(u_t), n) * n[0] + p_ * n[1]) * dS)

CD, CL, TT = [], [], []
for i in range(int(round(T / dt))):
    t += dt
    rhs(b1, L1, a1, bcu); s1.solve(b1, u_s.x.petsc_vec); u_s.x.scatter_forward()
    rhs(b2, L2, a2, bcp); s2.solve(b2, phi.x.petsc_vec); phi.x.scatter_forward()
    p_.x.petsc_vec.axpy(1.0, phi.x.petsc_vec); p_.x.scatter_forward()
    rhs(b3, L3); s3.solve(b3, u_.x.petsc_vec); u_.x.scatter_forward()
    u_n1.x.array[:] = u_n.x.array; u_n.x.array[:] = u_.x.array
    TT.append(t); CD.append(comm.allreduce(fem.assemble_scalar(drag), op=MPI.SUM)); CL.append(comm.allreduce(fem.assemble_scalar(lift), op=MPI.SUM))
CD, CL = np.array(CD), np.array(CL)
say(f"CD_max = {CD.max():.4f} at t = {TT[CD.argmax()]:.3f},  CL_max = {CL.max():.4f} at t = {TT[CL.argmax()]:.3f}")

In [ ]:
%%fenicsx -np 2
# TODO: Q4 your diagnostic copy and the corrected cell

> *your table for (a) and answers to (b), (c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …